# Padding dos Espectros — ELG, LRG, QSO

Le `{OBJ}spectra_all.h5` (estrutura `catalog/` + `spectra/{spec_id}/`), aplica os cortes de qualidade definidos em `02b_data_quality.ipynb`, alinha tudo numa grade log-lambda comum e grava `{OBJ}spectra_padded.h5`.

Estrutura de saida:
```
ml_dataset/X_spec   (N, L)  float32  — flux na grade global
ml_dataset/y        (N,)    float32  — redshift espectroscopico
wave_global         (L,)    float64  — grade comum em A
catalog/            metadados filtrados (zwarning, sn_median, ...)
```

## 1. Setup

In [1]:
import sys, time, os
from pathlib import Path
import numpy as np, h5py
from tqdm import tqdm

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, OBJECT_TYPES

## 2. Parametros

In [2]:
# Grade log-lambda padrao do SDSS
DLOGLAM      = 1e-4
TOL_DLOGLAM  = 1e-6
WAVE_MAX_MIN = 9000.0   # descarta espectros que nao chegam ate >= 9000 A

# Cortes de qualidade.
# Diagnostico em 02b_data_quality mostrou que zerr/(1+z) ja eh ~1e-4 mesmo em
# S/N baixo (curva quase plana em snr_vs_zerr.png), entao o unico corte que
# de fato remove targets duvidosos eh zwarning==0. Os demais sao mantidos
# como guard-rails afrouxados — nao removem quase nada e protegem contra
# valores patologicos (zerr explodido, sn_median=0/NaN).
USE_QUALITY_CUTS = True
ZWARNING_OK      = 0
ZERR_REL_MAX     = 1e-3   # zerr / (1 + z) — protege contra outliers
DELTACHI2_MIN    = 9.0
SNR_MIN          = 0.0    # afrouxado: S/N nao prediz qualidade do z neste regime

# Para teste rapido: limita a um subconjunto aleatorio
N_TESTE = None   # ou um int (e.g. 5000)
SEED    = 42

## 3. Funcoes auxiliares

In [3]:
def _to_str(x):
    return x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x)

def calcular_dloglam(wave):
    return float(np.diff(np.log10(wave)).mean())

def aplicar_padding(wave, flux, wave_global, dloglam=DLOGLAM):
    """Mapeia cada pixel na grade log-lambda global. Bordas/buracos = 0."""
    loglam_start = np.log10(wave_global[0])
    out = np.zeros(len(wave_global), dtype=np.float32)
    idx = np.round((np.log10(wave) - loglam_start) / dloglam).astype(int)
    ok  = (idx >= 0) & (idx < len(wave_global))
    out[idx[ok]] = flux[ok]
    return out

def quality_mask(cat):
    """Mascara booleana baseada nos cortes definidos nos parametros."""
    n = len(cat["redshift"])
    m = np.ones(n, dtype=bool)
    if not USE_QUALITY_CUTS:
        return m
    if "zwarning" in cat:
        m &= cat["zwarning"] == ZWARNING_OK
    if "zerr" in cat:
        rel = cat["zerr"] / (1 + cat["redshift"])
        m &= np.isfinite(rel) & (rel < ZERR_REL_MAX)
    if "deltachi2" in cat:
        m &= cat["deltachi2"] > DELTACHI2_MIN
    if "sn_median" in cat:
        m &= cat["sn_median"] > SNR_MIN
    return m

## 4. Pipeline por classe

In [4]:
def build_padded_for_object(obj):
    paths = paths_for(obj)
    raw   = paths["spectra_h5"]
    out   = raw.with_name(f"{obj}spectra_padded.h5")
    print(f"\n{'='*60}\n[{obj}] {raw}\n   -> {out}\n{'='*60}")
    out.parent.mkdir(parents=True, exist_ok=True)

    # 4.1 Catalogo + cortes de qualidade
    with h5py.File(raw, "r") as f:
        cat = {k: f["catalog"][k][:] for k in f["catalog"].keys()}
        spec_keys_all = set(f["spectra"].keys())
    spec_ids_cat = np.array([_to_str(s) for s in cat["spec_id"]])

    qmask = quality_mask(cat)
    print(f"[{obj}] cortes de qualidade: {qmask.sum():,}/{len(qmask):,} "
          f"({qmask.mean()*100:.1f}%)")

    # 4.2 Subamostragem opcional
    sel_idx = np.where(qmask)[0]
    if N_TESTE is not None and N_TESTE < len(sel_idx):
        rng = np.random.default_rng(SEED)
        sel_idx = np.sort(rng.choice(sel_idx, N_TESTE, replace=False))
        print(f"[{obj}] MODO TESTE: {len(sel_idx):,} aleatorios")

    # Garante que cada spec_id existe em /spectra/
    keys_sel = spec_ids_cat[sel_idx]
    in_h5    = np.array([k in spec_keys_all for k in keys_sel])
    if not in_h5.all():
        print(f"[{obj}] AVISO: {(~in_h5).sum()} spec_id sem espectro em /spectra/. Removendo.")
    keys_sel = keys_sel[in_h5]
    sel_idx  = sel_idx[in_h5]

    # 4.3 Le metadados de cada espectro (wave[0], wave[-1], dloglam)
    print(f"[{obj}] lendo metadados de {len(keys_sel):,} espectros...")
    t0 = time.time()
    wave_min = np.zeros(len(keys_sel))
    wave_max = np.zeros(len(keys_sel))
    dlog     = np.zeros(len(keys_sel))
    with h5py.File(raw, "r") as f:
        for i, k in enumerate(tqdm(keys_sel, desc=f"{obj} meta")):
            w = f[f"spectra/{k}/wave"][:]
            wave_min[i] = w[0]
            wave_max[i] = w[-1]
            dlog[i]     = calcular_dloglam(w)
    print(f"[{obj}] meta lido em {time.time()-t0:.1f}s")

    # 4.4 Filtro de grade
    m_grade = np.abs(dlog - DLOGLAM) <= TOL_DLOGLAM
    m_range = wave_max >= WAVE_MAX_MIN
    m_final = m_grade & m_range
    print(f"[{obj}] dloglam padrao    : {m_grade.sum():,} ({m_grade.mean()*100:.1f}%)")
    print(f"[{obj}] wave_max>={WAVE_MAX_MIN:.0f} A : {m_range.sum():,} ({m_range.mean()*100:.1f}%)")
    print(f"[{obj}] aprovados (final) : {m_final.sum():,} ({m_final.mean()*100:.1f}%)")

    keys_final    = keys_sel[m_final]
    cat_idx_final = sel_idx[m_final]
    N = len(keys_final)
    if N == 0:
        print(f"[{obj}] nenhum espectro aprovado. Pulando.")
        return

    # 4.5 Grade global
    loglam_start = np.log10(wave_min[m_final].min())
    loglam_end   = np.log10(wave_max[m_final].max())
    grade        = np.arange(loglam_start, loglam_end + DLOGLAM, DLOGLAM)
    wave_global  = 10 ** grade
    L = len(wave_global)
    print(f"[{obj}] grade global: [{wave_global[0]:.2f}, {wave_global[-1]:.2f}] A  L={L}")
    print(f"[{obj}] shape final do dataset: ({N}, {L})")

    # 4.6 Escreve HDF5 padded
    t0 = time.time()
    with h5py.File(raw, "r") as f_in, h5py.File(out, "w") as f_out:
        f_out.create_dataset("wave_global", data=wave_global)

        ds_X = f_out.create_dataset(
            "ml_dataset/X_spec",
            shape=(N, L), dtype="float32",
            compression="gzip", chunks=(64, L),
        )
        ds_y = f_out.create_dataset("ml_dataset/y", shape=(N,), dtype="float32")

        # Metadados filtrados (espelho de /catalog so com aprovados)
        cat_g = f_out.create_group("catalog")
        for k, arr in cat.items():
            cat_g.create_dataset(k, data=arr[cat_idx_final])

        # Atributos com a config usada
        f_out.attrs["dloglam"]       = DLOGLAM
        f_out.attrs["wave_max_min"]  = WAVE_MAX_MIN
        f_out.attrs["quality_cuts"]  = USE_QUALITY_CUTS
        f_out.attrs["snr_min"]       = SNR_MIN
        f_out.attrs["deltachi2_min"] = DELTACHI2_MIN
        f_out.attrs["zerr_rel_max"]  = ZERR_REL_MAX

        for i, k in enumerate(tqdm(keys_final, desc=f"{obj} pad")):
            wave = f_in[f"spectra/{k}/wave"][:]
            flux = f_in[f"spectra/{k}/flux"][:]
            ds_X[i] = aplicar_padding(wave, flux, wave_global)
            ds_y[i] = float(cat["redshift"][cat_idx_final[i]])

    print(f"[{obj}] gravado em {(time.time()-t0)/60:.1f} min  "
          f"({os.path.getsize(out)/1024**3:.2f} GB)")

## 5. Aplicar para os 3 objetos

In [5]:
for obj in OBJECT_TYPES:
    build_padded_for_object(obj)


[ELG] /home/thalita/spec_z_ml/data/processed/ELG/ELGspectra_all.h5
   -> /home/thalita/spec_z_ml/data/processed/ELG/ELGspectra_padded.h5
[ELG] cortes de qualidade: 169,189/169,192 (100.0%)
[ELG] lendo metadados de 169,189 espectros...


ELG meta: 100%|██████████| 169189/169189 [00:39<00:00, 4332.61it/s]


[ELG] meta lido em 39.1s
[ELG] dloglam padrao    : 166,935 (98.7%)
[ELG] wave_max>=9000 A : 169,055 (99.9%)
[ELG] aprovados (final) : 166,801 (98.6%)
[ELG] grade global: [3569.44, 10375.28] A  L=4635
[ELG] shape final do dataset: (166801, 4635)


ELG pad: 100%|██████████| 166801/166801 [02:38<00:00, 1050.29it/s]


[ELG] gravado em 2.7 min  (2.65 GB)

[LRG] /home/thalita/spec_z_ml/data/processed/LRG/LRGspectra_all.h5
   -> /home/thalita/spec_z_ml/data/processed/LRG/LRGspectra_padded.h5
[LRG] cortes de qualidade: 132,971/167,635 (79.3%)
[LRG] lendo metadados de 132,971 espectros...


LRG meta: 100%|██████████| 132971/132971 [00:37<00:00, 3592.71it/s]


[LRG] meta lido em 37.1s
[LRG] dloglam padrao    : 131,070 (98.6%)
[LRG] wave_max>=9000 A : 132,853 (99.9%)
[LRG] aprovados (final) : 130,952 (98.5%)
[LRG] grade global: [3547.32, 10403.99] A  L=4674
[LRG] shape final do dataset: (130952, 4674)


LRG pad: 100%|██████████| 130952/130952 [02:05<00:00, 1042.59it/s]


[LRG] gravado em 2.1 min  (2.08 GB)


## 6. Verificacao

In [6]:
for obj in OBJECT_TYPES:
    p = paths_for(obj)["spectra_h5"].with_name(f"{obj}spectra_padded.h5")
    if not p.exists():
        print(f"[{obj}] FALTANDO: {p}")
        continue
    with h5py.File(p, "r") as f:
        N = f["ml_dataset/y"].shape[0]
        L = f["ml_dataset/X_spec"].shape[1]
        wg = f["wave_global"][:]
        sample = f["ml_dataset/X_spec"][:200]
        zeros_pct = (sample == 0).mean() * 100
    print(f"[{obj}] N={N:,}  L={L}  wave=[{wg[0]:.0f},{wg[-1]:.0f}] A  "
          f"zeros(amostra)={zeros_pct:.1f}%  ({p.stat().st_size/1e9:.1f} GB)")

[ELG] N=166,801  L=4635  wave=[3569,10375] A  zeros(amostra)=1.1%  (2.8 GB)
[LRG] N=130,952  L=4674  wave=[3547,10404] A  zeros(amostra)=2.2%  (2.2 GB)
